# 01 - AIMU

This is the first notebook in the **Agent Frameworks** series. Every notebook in the series
implements the *same* scientific use case so you can compare frameworks side by side; only the
framework changes. See [README.md](../README.md) for the full series and the shared design.

## The use case: agentic literature research

Given a research question, the agent:

1. **Searches live sources** (medRxiv + bioRxiv preprints via the Europe PMC keyword API),
2. **Confirms relevance** of each hit and saves only the matches into a **local document store**
   (a curated, reusable corpus, gated so off-topic papers never land),
3. **Synthesizes** a cited answer from the curated corpus, and
4. **Evaluates and loops**: a critic reviews the synthesis and feeds gaps back for another round.

Everything runs on a **local model** through [Ollama](https://ollama.com).

## What AIMU brings

[`aimu`](https://github.com/) wraps a model client in an agentic loop (`Agent`) and composes
workflows like the generate-evaluate loop with `EvaluatorOptimizer`. This notebook uses three
capabilities built for exactly this kind of task:

- **`ToolContext`** — tools declare a `ctx: ToolContext[Deps]` parameter and the agent injects shared
  state (the document store, the DOI cache) at call time, hidden from the model. No module globals.
- **`EvaluatorOptimizer(verdict_schema=...)`** — the critic returns a typed `Verdict(passed, feedback)`
  via structured output instead of matching a magic `PASS` string.
- **`aimu.pretty_print`** — renders a streamed run (tool calls + text) without a hand-written loop.

The next notebook, **02 - genscai**, builds the same workflow from the project's
[`genscai.corpus.Corpus`](../../genscai/corpus.py) helper, which packages these tools for you.

> **Script version:** [`scripts/01_aimu.py`](../scripts/01_aimu.py) runs this same workflow from the command line.

## 01 - Setup

Requires a running Ollama server with a tool-capable model and an embedding model:

```bash
ollama pull qwen3.6:27b
ollama pull nomic-embed-text
```

`qwen3.6:27b` is large but reliably handles the multi-step tool use this task needs. A smaller
model such as `qwen3.5:9b` runs faster but is less consistent at open-ended research.

In [ ]:
import os
from dataclasses import dataclass, field

import dotenv
from pydantic import BaseModel

import aimu
from aimu import ToolContext
from aimu.agents import Agent, EvaluatorOptimizer
from aimu.memory import DocumentStore

from genscai import research, paths

dotenv.load_dotenv(paths.root / ".env")

MODEL = f"ollama:{os.environ.get('GENSCAI_AGENT_MODEL', 'qwen3.6:27b')}"

RESEARCH_QUESTION = (
    "What interventions and control strategies have recent preprints proposed or evaluated for "
    "dengue outbreaks, and what evidence do they report?"
)

## 02 - The shared research tools

`genscai.research` provides the framework-agnostic search/fetch functions used by *every* notebook
in this series. They query medRxiv and bioRxiv preprints through the
[Europe PMC](https://europepmc.org/) keyword API (which indexes both servers and returns abstracts
reliably). A quick look at what one search returns:

In [ ]:
hits = research.search_medrxiv("dengue vaccination", max_results=3)
for h in hits:
    print(h["date"], "-", h["title"][:80])
    print("   ", h["doi"])

## 03 - The document store and agent tools

AIMU's `DocumentStore` is the *native* store for this notebook (persistent, file-backed). We expose
three tools to the agent. The relevance decision is the agent's own reasoning: it reads each
abstract and calls `save_relevant_paper` **only** for matches, so the store stays curated.

The tools need shared state — the store, and a cache of search hits so a paper can be saved by DOI
alone (without echoing long abstracts back through the model). Rather than module globals, each tool
declares a `ctx: ToolContext[ResearchDeps]` parameter: AIMU injects it at call time from the agent's
`deps=`, and it never appears in the model-facing schema.

In [ ]:
@dataclass
class ResearchDeps:
    """Shared state the tools need, injected via ToolContext instead of module globals."""

    store: DocumentStore
    seen: dict[str, dict] = field(default_factory=dict)


@aimu.tool
def search_preprints(ctx: ToolContext[ResearchDeps], query: str) -> str:
    """Search medRxiv and bioRxiv preprints. Returns each hit's DOI, title, date, and abstract."""
    try:
        results = research.search_medrxiv(query, max_results=5) + research.search_biorxiv(query, max_results=3)
    except Exception as exc:
        return f"Search temporarily unavailable ({exc}). Try again shortly or rephrase the query."
    if not results:
        return "No results found."
    blocks = []
    for article in results:
        ctx.deps.seen[article["doi"]] = article
        blocks.append(
            f"DOI: {article['doi']}\nTitle: {article['title']}\nDate: {article['date']}\n"
            f"Abstract: {(article['abstract'] or '')[:600]}"
        )
    return "\n\n".join(blocks)


@aimu.tool
def save_relevant_paper(ctx: ToolContext[ResearchDeps], doi: str) -> str:
    """Save a paper you have confirmed is relevant to the research question, identified by its DOI."""
    article = ctx.deps.seen.get(doi)
    if not article:
        return f"Unknown DOI {doi}. Search for it first so its details are available."
    content = (
        f"# {article['title']}\n\nDOI: {doi}\nAuthors: {article['authors']}\n"
        f"Date: {article['date']}\nURL: {article['url']}\n\n{article['abstract']}"
    )
    ctx.deps.store.write(f"/{doi.replace('/', '_')}.md", content)
    return f"Saved: {article['title']}"


@aimu.tool
def read_saved_papers(ctx: ToolContext[ResearchDeps]) -> str:
    """Read every paper saved in the local document store, to ground your synthesis."""
    saved = ctx.deps.store.list_paths()
    if not saved:
        return "No papers saved yet."
    return "\n\n---\n\n".join(ctx.deps.store.read(p) for p in saved)


deps = ResearchDeps(store=DocumentStore(persist_path=str(paths.output / "literature_research" / "aimu_store")))

## 04 - The researcher agent

An `Agent` is a model client wrapped in a loop: it calls tools until it produces a turn with no
tool calls (or hits `max_iterations`). We pass `deps=deps` so the tools' injected `ctx.deps` resolves
to our store and cache. `aimu.pretty_print` renders the streamed run — flagging each tool call and
streaming the text — so we can watch it search, gate relevance, store matches, and synthesize.
`final_answer_prompt` guarantees a written synthesis even if the loop hits its iteration cap
mid-tool-use.

In [ ]:
researcher = Agent(
    aimu.client(MODEL),
    system_message=(
        "You are a research assistant building a cited literature review grounded in a curated corpus.\n"
        "Workflow:\n"
        "1. Call read_saved_papers first to see what is already curated.\n"
        "2. Use search_preprints to find candidate papers. Always include the key topic term from the "
        "question (e.g. 'dengue') in every search query so results stay on-topic.\n"
        "3. For EACH candidate, judge whether its abstract is relevant to the question. If it is plausibly "
        "relevant, call save_relevant_paper(doi); err toward saving rather than discarding. Skip only "
        "clearly off-topic hits.\n"
        "4. Before writing any synthesis you MUST call read_saved_papers again, and base the synthesis "
        "ONLY on the papers it returns, citing each by title and DOI. The store already contains relevant "
        "papers, so never claim no papers were found. Be concise."
    ),
    tools=[search_preprints, save_relevant_paper, read_saved_papers],
    deps=deps,
    max_iterations=12,
    reset_messages_on_run=True,
    final_answer_prompt=(
        "Call read_saved_papers, then write the final cited synthesis based only on those saved papers. "
        "Do not claim there are no papers if the store is non-empty."
    ),
    name="researcher",
)

aimu.pretty_print(researcher.run(RESEARCH_QUESTION, stream=True))

## 05 - Add an evaluator: the self-correcting loop

`EvaluatorOptimizer` runs a generator, has a critic review the output, and loops with the critic's
feedback until it accepts or `max_rounds` is reached. Here the critic returns a typed
`Verdict(passed, feedback)` — `verdict_schema=Verdict` reads `passed` to decide acceptance and
`feedback` to drive the revision, so there's no brittle `PASS` string-matching and the critic prompt
is written for a human. The document store persists across rounds, so each revision builds on the
corpus curated so far.

In [ ]:
class Verdict(BaseModel):
    passed: bool
    feedback: str = ""


critic = Agent(
    aimu.client(MODEL),
    system_message=(
        "You review a literature synthesis built from a curated corpus of saved preprints. Set "
        "passed=true when it answers the question, cites specific papers (title + DOI), and reports "
        "evidence from them; otherwise set passed=false and put the single most important missing piece "
        "in feedback. Do not demand topics outside the saved corpus, and do not penalize a focused answer."
    ),
    max_iterations=1,
    reset_messages_on_run=True,
    name="critic",
)

review = EvaluatorOptimizer(generator=researcher, evaluator=critic, max_rounds=2, verdict_schema=Verdict)

synthesis = review.run(RESEARCH_QUESTION)
print(synthesis)

## 06 - Inspect the curated corpus

The agent built a local, reusable corpus of vetted papers. Because the store is gated by the
relevance check, every document here was judged relevant to the question.

In [ ]:
print("Saved papers:")
for path in deps.store.list_paths():
    print(" ", path)

print("\nFirst saved paper:\n")
print(deps.store.read(deps.store.list_paths()[0]))

## 07 - AIMU notes

- **Tools**: plain functions with `@aimu.tool`; the docstring becomes the tool description and type
  hints become the schema. A `ctx: ToolContext[Deps]` parameter is injected by the agent (from
  `deps=`) and hidden from the model, so tools reach shared state without module globals.
- **Document store**: `aimu.memory.DocumentStore` is built in and persistent. AIMU also ships
  `make_memory_tools`/`make_retrieval_tool` (in `aimu.tools.builtin`) and a vector-backed
  `SemanticMemoryStore` if you want semantic retrieval instead of a curated file store.
- **The feedback loop**: `EvaluatorOptimizer` expresses generate→evaluate→revise as a first-class
  workflow object. `verdict_schema=` makes the critic return a typed `Verdict(passed, feedback)`
  via structured output (a `stop_when` predicate and the default `pass_keyword="PASS"` are also
  available).
- **Output**: `aimu.pretty_print` renders any streamed run and returns the generated text.
- **Local models**: `aimu.client("ollama:...")` is all that's needed; the agentic loop, streaming,
  and tool-calling work the same across providers.
- **Going further**: **02 - genscai** packages these three tools (and the DOI cache) into one
  `genscai.corpus.Corpus`, so an agent gets the whole search→gate→persist→read pattern from a single
  constructor.